# Project: Content Router

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangGraph roadmap.

You will build a content router that classifies input (poem, story, or joke) using structured output, routes to specialized generator nodes via conditional edges, and returns the generated content.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the State

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    content_type: str

## 4. Create the Classifier

Use `with_structured_output` and `Literal` types to guarantee valid routing keys.

In [ ]:
from pydantic import BaseModel
from typing import Literal
from langchain_openai import ChatOpenAI

class ContentType(BaseModel):
    """Classify the user request into a content type."""
    content_type: Literal["poem", "story", "joke"]

llm = ChatOpenAI(model="gpt-4o-mini")
classifier_llm = llm.with_structured_output(ContentType)

def classifier(state: State) -> dict:
    result = classifier_llm.invoke(state["messages"])
    return {"content_type": result.content_type}

test = classifier_llm.invoke([("human", "Tell me something funny about cats")])
print(f"Test classification: {test.content_type}")

## 5. Build the Generator Nodes

Each generator uses a tailored system prompt.

In [ ]:
def poem_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a poet. Write a short, evocative poem based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

def story_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a storyteller. Write a short, engaging story based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

def joke_generator(state: State) -> dict:
    response = llm.invoke([
        ("system", "You are a comedian. Write a funny joke based on the user's request."),
        *state["messages"],
    ])
    return {"messages": [response]}

## 6. Define the Routing Function

In [ ]:
def route_content(state: State) -> str:
    return state["content_type"]

## 7. Assemble and Compile the Graph

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)
graph.add_node("classifier", classifier)
graph.add_node("poem_generator", poem_generator)
graph.add_node("story_generator", story_generator)
graph.add_node("joke_generator", joke_generator)

graph.add_edge(START, "classifier")
graph.add_conditional_edges("classifier", route_content, {
    "poem": "poem_generator",
    "story": "story_generator",
    "joke": "joke_generator",
})
graph.add_edge("poem_generator", END)
graph.add_edge("story_generator", END)
graph.add_edge("joke_generator", END)

app = graph.compile()

## 8. Test the Router

In [ ]:
result = app.invoke({"messages": [("human", "Tell me a joke about programmers")]})
print(f"Content type: {result['content_type']}")
print(f"Output:\n{result['messages'][-1].content}")

In [ ]:
result = app.invoke({"messages": [("human", "Write a poem about the ocean at sunset")]})
print(f"Content type: {result['content_type']}")
print(f"Output:\n{result['messages'][-1].content}")

In [ ]:
result = app.invoke({"messages": [("human", "Tell me a story about a brave knight and a dragon")]})
print(f"Content type: {result['content_type']}")
print(f"Output:\n{result['messages'][-1].content}")

## What You Learned

1. **Structured output** with `Literal` types guarantees valid routing keys
2. **`add_conditional_edges`** cleanly separates routing from generation
3. Each **generator node** has its own system prompt and personality
4. The pattern **scales easily** — add a Literal value, a generator, and a mapping entry
5. This is a building block for any **classify-then-act** application